# PyTorch 常用層和損失函數詳解

本教程詳細介紹 PyTorch 中常用的神經網路層和損失函數,幫助你構建各種深度學習模型。

## 目錄
1. 線性層
2. 卷積層
3. 池化層
4. 正規化層
5. Dropout 層
6. 激活函數
7. 循環層
8. 注意力機制
9. 損失函數

**作者:** AI Learning Notes  
**最後更新:** 2025-01

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

print(f"PyTorch 版本: {torch.__version__}")

## 1. 線性層 (Linear Layer)

全連接層,也稱為密集層或線性層,是最基本的神經網路層。

In [ ]:
# 創建線性層
# nn.Linear(in_features, out_features, bias=True)
linear = nn.Linear(10, 5)

# 輸入: (batch_size, in_features)
x = torch.randn(32, 10)
output = linear(x)

print(f"輸入形狀: {x.shape}")
print(f"輸出形狀: {output.shape}")
print(f"權重形狀: {linear.weight.shape}")
print(f"偏置形狀: {linear.bias.shape}")

# 查看參數
print(f"\n參數數量: {sum(p.numel() for p in linear.parameters())}")

## 2. 卷積層 (Convolution Layer)

用於處理具有網格結構的資料,如圖像。

### 2.1 二維卷積 (Conv2d)

In [ ]:
# nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0)
conv2d = nn.Conv2d(
    in_channels=3,      # 輸入通道數 (RGB圖像為3)
    out_channels=16,    # 輸出通道數 (濾波器數量)
    kernel_size=3,      # 卷積核大小 3x3
    stride=1,           # 步長
    padding=1           # 填充
)

# 輸入: (batch_size, channels, height, width)
x = torch.randn(32, 3, 224, 224)
output = conv2d(x)

print(f"輸入形狀: {x.shape}")
print(f"輸出形狀: {output.shape}")
print(f"權重形狀: {conv2d.weight.shape}")
print(f"參數數量: {sum(p.numel() for p in conv2d.parameters())}")

### 2.2 一維卷積 (Conv1d) - 用於序列資料

In [ ]:
# 用於文本、音頻等序列資料
conv1d = nn.Conv1d(
    in_channels=128,     # 輸入特徵維度
    out_channels=256,    # 輸出特徵維度
    kernel_size=3,       # 卷積核大小
    padding=1
)

# 輸入: (batch_size, channels, length)
x = torch.randn(32, 128, 100)  # 32個序列,每個100長度,128維特徵
output = conv1d(x)

print(f"Conv1d 輸入: {x.shape}")
print(f"Conv1d 輸出: {output.shape}")

### 2.3 轉置卷積 (ConvTranspose2d) - 上採樣

In [ ]:
# 用於圖像生成、分割等任務
conv_transpose = nn.ConvTranspose2d(
    in_channels=64,
    out_channels=32,
    kernel_size=4,
    stride=2,
    padding=1
)

x = torch.randn(8, 64, 28, 28)
output = conv_transpose(x)

print(f"轉置卷積輸入: {x.shape}")
print(f"轉置卷積輸出: {output.shape}  # 尺寸翻倍")

## 3. 池化層 (Pooling Layer)

降低特徵圖的空間維度,減少計算量並提取重要特徵。

In [ ]:
x = torch.randn(32, 64, 56, 56)

# 最大池化
maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
output_max = maxpool(x)
print(f"MaxPool2d 輸出: {output_max.shape}")

# 平均池化
avgpool = nn.AvgPool2d(kernel_size=2, stride=2)
output_avg = avgpool(x)
print(f"AvgPool2d 輸出: {output_avg.shape}")

# 自適應池化 (輸出固定大小)
adaptive_avgpool = nn.AdaptiveAvgPool2d((1, 1))  # 全局平均池化
output_adaptive = adaptive_avgpool(x)
print(f"AdaptiveAvgPool2d 輸出: {output_adaptive.shape}")

# 可以直接展平用於全連接層
flattened = output_adaptive.view(output_adaptive.size(0), -1)
print(f"展平後: {flattened.shape}")

## 4. 正規化層 (Normalization Layer)

正規化層有助於加速訓練並提高模型穩定性。

### 4.1 批次正規化 (Batch Normalization)

In [ ]:
# BatchNorm2d: 用於卷積層之後
bn2d = nn.BatchNorm2d(num_features=64)
x = torch.randn(32, 64, 28, 28)
output = bn2d(x)
print(f"BatchNorm2d 輸出: {output.shape}")
print(f"均值: {output.mean().item():.4f}")
print(f"標準差: {output.std().item():.4f}")

# BatchNorm1d: 用於全連接層之後
bn1d = nn.BatchNorm1d(num_features=128)
x = torch.randn(32, 128)
output = bn1d(x)
print(f"\nBatchNorm1d 輸出: {output.shape}")

### 4.2 Layer Normalization

In [ ]:
# LayerNorm: 常用於 Transformer
ln = nn.LayerNorm(normalized_shape=512)
x = torch.randn(32, 10, 512)  # (batch, sequence, features)
output = ln(x)
print(f"LayerNorm 輸出: {output.shape}")
print(f"每個樣本的均值: {output[0].mean().item():.6f}")
print(f"每個樣本的標準差: {output[0].std().item():.6f}")

### 4.3 Group Normalization

In [ ]:
# GroupNorm: 介於 BatchNorm 和 LayerNorm 之間
gn = nn.GroupNorm(num_groups=8, num_channels=64)
x = torch.randn(32, 64, 28, 28)
output = gn(x)
print(f"GroupNorm 輸出: {output.shape}")

## 5. Dropout 層

正則化技術,隨機丟棄一部分神經元,防止過擬合。

In [ ]:
# Dropout: 用於全連接層
dropout = nn.Dropout(p=0.5)  # 50% 的神經元被丟棄
x = torch.ones(10, 10)

# 訓練模式
dropout.train()
output_train = dropout(x)
print("訓練模式 (部分為0):")
print(output_train[:5, :5])

# 推理模式
dropout.eval()
output_eval = dropout(x)
print("\n推理模式 (全部保留):")
print(output_eval[:5, :5])

# Dropout2d: 用於卷積層,丟棄整個通道
dropout2d = nn.Dropout2d(p=0.2)
x = torch.randn(8, 64, 28, 28)
dropout2d.train()
output = dropout2d(x)
print(f"\nDropout2d 輸出: {output.shape}")

## 6. 激活函數 (Activation Functions)

引入非線性,使神經網路能夠學習複雜的函數。

In [ ]:
x = torch.linspace(-3, 3, 100)

# 常用激活函數
activations = {
    'ReLU': nn.ReLU(),
    'LeakyReLU': nn.LeakyReLU(negative_slope=0.1),
    'PReLU': nn.PReLU(),
    'ELU': nn.ELU(),
    'GELU': nn.GELU(),
    'Sigmoid': nn.Sigmoid(),
    'Tanh': nn.Tanh(),
    'Softplus': nn.Softplus(),
    'SiLU (Swish)': nn.SiLU()
}

# 可視化
fig, axes = plt.subplots(3, 3, figsize=(12, 10))
axes = axes.flatten()

for idx, (name, activation) in enumerate(activations.items()):
    y = activation(x)
    axes[idx].plot(x.numpy(), y.detach().numpy())
    axes[idx].grid(True)
    axes[idx].set_title(name)
    axes[idx].axhline(y=0, color='k', linewidth=0.5)
    axes[idx].axvline(x=0, color='k', linewidth=0.5)

plt.tight_layout()
plt.show()

# 使用示例
print("\n激活函數使用建議:")
print("- ReLU: 最常用,簡單高效")
print("- LeakyReLU: 解決 ReLU 的死神經元問題")
print("- GELU: Transformer 中常用")
print("- SiLU (Swish): EfficientNet 等模型中使用")
print("- Sigmoid/Tanh: 主要用於輸出層或門控機制")

## 7. 循環層 (Recurrent Layer)

用於處理序列資料。

### 7.1 LSTM

In [ ]:
# LSTM
lstm = nn.LSTM(
    input_size=128,      # 輸入特徵維度
    hidden_size=256,     # 隱藏層維度
    num_layers=2,        # 層數
    batch_first=True,    # 輸入形狀 (batch, seq, feature)
    dropout=0.2,         # Dropout
    bidirectional=True   # 雙向
)

# 輸入: (batch, sequence_length, input_size)
x = torch.randn(32, 100, 128)
output, (hidden, cell) = lstm(x)

print(f"LSTM 輸入: {x.shape}")
print(f"LSTM 輸出: {output.shape}")
print(f"隱藏狀態: {hidden.shape}")
print(f"細胞狀態: {cell.shape}")
print(f"\n雙向LSTM輸出維度 = hidden_size * 2 = {256 * 2}")

### 7.2 GRU

In [ ]:
# GRU: 比 LSTM 更簡單,參數更少
gru = nn.GRU(
    input_size=128,
    hidden_size=256,
    num_layers=2,
    batch_first=True,
    bidirectional=True
)

x = torch.randn(32, 100, 128)
output, hidden = gru(x)

print(f"GRU 輸入: {x.shape}")
print(f"GRU 輸出: {output.shape}")
print(f"隱藏狀態: {hidden.shape}")

## 8. 注意力機制 (Attention)

Transformer 的核心組件。

In [ ]:
# MultiheadAttention
attention = nn.MultiheadAttention(
    embed_dim=512,       # 特徵維度
    num_heads=8,         # 注意力頭數
    dropout=0.1,
    batch_first=True
)

# 輸入: (batch, sequence_length, embed_dim)
x = torch.randn(32, 100, 512)
output, attention_weights = attention(x, x, x)

print(f"注意力輸入: {x.shape}")
print(f"注意力輸出: {output.shape}")
print(f"注意力權重: {attention_weights.shape}")

# TransformerEncoderLayer
encoder_layer = nn.TransformerEncoderLayer(
    d_model=512,
    nhead=8,
    dim_feedforward=2048,
    dropout=0.1,
    batch_first=True
)

x = torch.randn(32, 100, 512)
output = encoder_layer(x)
print(f"\nTransformerEncoderLayer 輸出: {output.shape}")

## 9. 損失函數 (Loss Functions)

用於訓練模型的目標函數。

### 9.1 分類任務

In [ ]:
# 交叉熵損失 (最常用)
criterion = nn.CrossEntropyLoss()

# 模型輸出 (未經過 softmax 的 logits)
predictions = torch.randn(32, 10)  # 32個樣本, 10個類別
targets = torch.randint(0, 10, (32,))  # 真實標籤

loss = criterion(predictions, targets)
print(f"CrossEntropyLoss: {loss.item():.4f}")

# 二元交叉熵 (二分類)
bce_loss = nn.BCELoss()
predictions = torch.sigmoid(torch.randn(32, 1))
targets = torch.randint(0, 2, (32, 1)).float()
loss = bce_loss(predictions, targets)
print(f"BCELoss: {loss.item():.4f}")

# BCEWithLogitsLoss (數值更穩定)
bce_logits_loss = nn.BCEWithLogitsLoss()
predictions = torch.randn(32, 1)
targets = torch.randint(0, 2, (32, 1)).float()
loss = bce_logits_loss(predictions, targets)
print(f"BCEWithLogitsLoss: {loss.item():.4f}")

# 負對數似然損失
nll_loss = nn.NLLLoss()
predictions = F.log_softmax(torch.randn(32, 10), dim=1)
targets = torch.randint(0, 10, (32,))
loss = nll_loss(predictions, targets)
print(f"NLLLoss: {loss.item():.4f}")

### 9.2 回歸任務

In [ ]:
# 均方誤差損失 (MSE)
mse_loss = nn.MSELoss()
predictions = torch.randn(32, 1)
targets = torch.randn(32, 1)
loss = mse_loss(predictions, targets)
print(f"MSELoss: {loss.item():.4f}")

# 平均絕對誤差損失 (MAE)
mae_loss = nn.L1Loss()
loss = mae_loss(predictions, targets)
print(f"L1Loss (MAE): {loss.item():.4f}")

# Smooth L1 Loss (Huber Loss)
smooth_l1_loss = nn.SmoothL1Loss()
loss = smooth_l1_loss(predictions, targets)
print(f"SmoothL1Loss: {loss.item():.4f}")

### 9.3 進階損失函數

In [ ]:
# Focal Loss (處理類別不平衡)
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

focal_loss = FocalLoss(alpha=1, gamma=2)
predictions = torch.randn(32, 10)
targets = torch.randint(0, 10, (32,))
loss = focal_loss(predictions, targets)
print(f"FocalLoss: {loss.item():.4f}")

# Cosine Embedding Loss (相似度學習)
cosine_loss = nn.CosineEmbeddingLoss()
input1 = torch.randn(32, 128)
input2 = torch.randn(32, 128)
targets = torch.ones(32)  # 1 表示相似, -1 表示不相似
loss = cosine_loss(input1, input2, targets)
print(f"CosineEmbeddingLoss: {loss.item():.4f}")

# Triplet Loss (度量學習)
triplet_loss = nn.TripletMarginLoss(margin=1.0)
anchor = torch.randn(32, 128)
positive = torch.randn(32, 128)
negative = torch.randn(32, 128)
loss = triplet_loss(anchor, positive, negative)
print(f"TripletMarginLoss: {loss.item():.4f}")

### 9.4 損失函數選擇指南

In [ ]:
print("損失函數選擇指南:\n")
print("分類任務:")
print("  - 多分類: CrossEntropyLoss")
print("  - 二分類: BCEWithLogitsLoss")
print("  - 類別不平衡: FocalLoss, 加權CrossEntropyLoss")
print("  - 多標籤: BCEWithLogitsLoss")
print()
print("回歸任務:")
print("  - 標準回歸: MSELoss")
print("  - 對異常值魯棒: L1Loss, SmoothL1Loss")
print("  - 百分比誤差: MAPE (自定義)")
print()
print("特殊任務:")
print("  - 相似度學習: CosineEmbeddingLoss")
print("  - 度量學習: TripletMarginLoss")
print("  - 對比學習: ContrastiveLoss (自定義)")
print("  - 生成任務: MSELoss, L1Loss")
print("  - 對抗訓練: BCELoss, Hinge Loss")

## 10. 實戰:構建完整模型

In [ ]:
class CompleteModel(nn.Module):
    """展示各種層的使用"""
    
    def __init__(self, num_classes=10):
        super(CompleteModel, self).__init__()
        
        # 卷積塊
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu1 = nn.ReLU(inplace=True)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(128)
        self.relu2 = nn.ReLU(inplace=True)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # 全局平均池化
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        
        # 全連接層
        self.fc1 = nn.Linear(128, 256)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(256, num_classes)
    
    def forward(self, x):
        # 卷積塊 1
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu1(x)
        x = self.pool1(x)
        
        # 卷積塊 2
        x = self.conv2(x)
        x = self.bn2(x)
        x = self.relu2(x)
        x = self.pool2(x)
        
        # 全局池化
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)
        
        # 全連接層
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

# 創建模型
model = CompleteModel(num_classes=10)
print("模型結構:")
print(model)

# 測試前向傳播
x = torch.randn(4, 3, 32, 32)
output = model(x)
print(f"\n輸入: {x.shape}")
print(f"輸出: {output.shape}")

# 計算參數量
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\n總參數: {total_params:,}")
print(f"可訓練參數: {trainable_params:,}")

## 總結

本教程涵蓋了 PyTorch 中最常用的層和損失函數:

### 常用層
1. **Linear**: 全連接層
2. **Conv2d/Conv1d**: 卷積層
3. **MaxPool2d/AvgPool2d**: 池化層
4. **BatchNorm/LayerNorm**: 正規化層
5. **Dropout**: 正則化
6. **ReLU/GELU**: 激活函數
7. **LSTM/GRU**: 循環層
8. **MultiheadAttention**: 注意力機制

### 損失函數
1. **CrossEntropyLoss**: 多分類
2. **BCEWithLogitsLoss**: 二分類
3. **MSELoss**: 回歸
4. **FocalLoss**: 類別不平衡
5. **TripletMarginLoss**: 度量學習

### 使用建議
1. 卷積層後接 BatchNorm 和 ReLU
2. 全連接層前使用 Dropout
3. 根據任務選擇合適的損失函數
4. 注意各層的輸入輸出形狀
5. 訓練和推理時 BatchNorm/Dropout 的行為不同

### 參考資源
- [torch.nn Documentation](https://pytorch.org/docs/stable/nn.html)
- [PyTorch Tutorials](https://pytorch.org/tutorials/)
- [Papers with Code](https://paperswithcode.com/)